2025-06-20<br>
RuiGao@ucmerced.edu<br>
Workshop at Logan

# TSEB Model Batch Processing
Supports:
- TSEB-PT (Priestley-Taylor based)
- TSEB-2T (Dual-component canopy-soil temperatures)

Each row in the input Excel file corresponds to a single model run with site-specific inputs.<br>
The provide Excel file is just an example, which does not reflect the reality (for model test purpose).<br>
Further development is needed.
 modifyed sac version
 

In [ ]:
from pyTSEB.TSEB import TSEB_2T
from pyTSEB.TSEB import TSEB_PT
from pyTSEB.TSEB import TSEB_SW
from pyTSEB.TSEB import TSEB_PM
from pyTSEB.TSEB import DTD
from pyTSEB.TSEB import KB_1_DEFAULT

import pyTSEB.net_radiation as py_net_rad
import pyTSEB.resistances as py_net_res
import pyTSEB.meteo_utils as py_meteo
import pyTSEB.clumping_index as py_clumping

import os
import pandas as pd
import numpy as np
from osgeo import gdal

excel_table = r"C:\Copy_Paste\4_TSEB_Loop\Workshop_Logan_Weather_Support_Data.xlsx"
df = pd.read_excel(excel_table, header=1)
num_runs = len(df)
display(df)
print("There are totally", num_runs, "runs the model will run.")

In [4]:
# Define the cleaning function
# Any output beyond this range will be assigned -9999
def clean_band(data, vmin=-300, vmax=1000, nodata_val=-9999):
    data_cleaned = np.where((data < vmin) | (data > vmax), nodata_val, data)
    return data_cleaned

def run_tseb_model(model_type,
                   Tr_K=None, vza=None, T_C=None, T_S=None,
                   T_A_K=None, u=None, ea=None, p=1013,
                   Sn_C=None, Sn_S=None, L_dn=None,
                   LAI=None, h_C=None,
                   emis_C=None, emis_S=None,
                   z_0M=None, d_0=None, z_u=None, z_T=None,
                   leaf_width=0.1, z0_soil=0.01, alpha_PT=1.26,
                   x_LAD=1.0, f_c=1.0, f_g=1.0, w_C=1.0,
                   resistance_form=None, calcG_params=None, const_L=None,
                   kB=None, massman_profile=None, verbose=True):
    """
    Run either TSEB_PT or TSEB_2T model based on model_type input.

    Parameters:
        model_type (str): Either 'PT' or '2T' to select TSEB_PT or TSEB_2T.
        All other parameters are passed to the respective model.

    Returns:
        Model output as unpacked tuple
    """
    # from TSEB import TSEB_PT, TSEB_2T, KB_1_DEFAULT

    # Use default kB if not passed
    if kB is None:
        kB = KB_1_DEFAULT

    if model_type == 'PT':
        return TSEB_PT(
            Tr_K, vza, T_A_K, u, ea, p,
            Sn_C, Sn_S, L_dn, LAI, h_C,
            emis_C, emis_S, z_0M, d_0,
            z_u, z_T,
            leaf_width, z0_soil, alpha_PT,
            x_LAD, f_c, f_g, w_C,
            resistance_form, calcG_params, const_L,
            kB, massman_profile, verbose
        )
    elif model_type == '2T':
        return TSEB_2T(
            T_C, T_S, T_A_K, u, ea, p,
            Sn_C, Sn_S, L_dn, LAI, h_C,
            emis_C, emis_S, z_0M, d_0,
            z_u, z_T,
            leaf_width, z0_soil, alpha_PT,
            x_LAD, f_c, f_g, w_C,
            resistance_form, calcG_params, const_L,
            kB, massman_profile, verbose
        )
    else:
        raise ValueError("Invalid model_type. Use 'PT' for TSEB_PT or '2T' for TSEB_2T.")

In [ ]:
# Choose the model type: 
# Currently, only 'PT' and '2T' are involved
model_type_to_run = 'PT'  

for loopi in range(0,len(df)):    
    # loopi = 0
    # Input images
    LAI = df['LAI'][loopi]
    h_C = df['CH'][loopi]
    w_C = df['WH'][loopi]
    f_c = df['Fc'][loopi]
    # One-layer temperature
    temp_layer_1 = df['1LTemp'][loopi]
    # Two-layer temperature
    temp_layer_2 = df['2LTemp'][loopi]

    # Temperature reading for TSEB-PT model
    fid = gdal.Open(temp_layer_1, gdal.GA_ReadOnly)
    gt = fid.GetGeoTransform()
    proj = fid.GetProjection()
    Tr_K = fid.GetRasterBand(1).ReadAsArray()
    dims = Tr_K.shape  # (rows, cols)
    del fid

    # Temperature reading for TSEB-2T model
    # # Canopy temperature and soil temperature
    fid = gdal.Open(temp_layer_2, gdal.GA_ReadOnly)
    T_C = fid.GetRasterBand(1).ReadAsArray()
    T_S = fid.GetRasterBand(2).ReadAsArray()
    del fid

    # Canopy height
    fid = gdal.Open(h_C, gdal.GA_ReadOnly)
    h_C = fid.GetRasterBand(1).ReadAsArray()
    del fid

    # Canopy width over canopy height
    fid = gdal.Open(w_C, gdal.GA_ReadOnly)
    w_C = fid.GetRasterBand(1).ReadAsArray()
    del fid

    # LAI
    fid = gdal.Open(LAI, gdal.GA_ReadOnly)
    LAI = fid.GetRasterBand(1).ReadAsArray()
    del fid

    # LAI
    fid = gdal.Open(f_c, gdal.GA_ReadOnly)
    f_c = fid.GetRasterBand(1).ReadAsArray()
    del fid

    # Decimal values
    doy = df['DOY'][loopi]
    time = df['Decimal time'][loopi]
    lat = df['Y'][loopi]
    lon = df['X'][loopi]
    alt = df['altitude'][loopi]
    std_lon = df['std long'][loopi]
    z_u = df['sensor height'][loopi]
    z_t = df['sensor height'][loopi]

    vza = df['VZA'][loopi]
    T_A_K = df['air tmp'][loopi]
    u = df['Vw'][loopi]
    ea = df['VP'][loopi]
    p = df['p'][loopi]
    L_dn = df['Lin'][loopi]
    S_dn = df['shortwave'][loopi]
    [emis_C, emis_S] = [0.98, 0.95]
    f_g = df['f_g'][loopi]
    kB = df['kB'][loopi]
    
    # landcover type: 
    # [crops, grass, shrubs, 
    # conifer forests, broadleaved forests] = [11, 2, 5, 4, 3]
    landcover = 11

    # Spectral properties
    [rho_vis_C, rho_nir_C, 
    tau_vis_C, tau_nir_C, 
    rho_vis_S, rho_nir_S, 
    emis_C, emis_S] =[0.07, 0.32,
                    0.08, 0.33,
                    0.15, 0.25,
                    0.98, 0.95]

    # Surface properties
    [alpha_PT, x_LAD, 
    leaf_width, z0_soil] = [1.26, 1, 
                            0.1, 0.01]

    # Additional options
    [resistance_form, 
    KN_b, KN_c, KN_C_dash,
    calc_row, row_az, 
    G_form, G_constant,
    G_ratio, G_amp,
    G_phase, G_shape] = [0, 0.065,
                        0.0038, 90, 
                        0, 90,
                        2, 0.0,
                        df['G-ratio'][loopi], 0.35,
                        3.0, 24.0]

    # Calculate Sn_C and Sn_S
    sza, saa = py_meteo.calc_sun_angles(lat, lon, std_lon, doy, time)
    sza = np.full_like(S_dn, float(sza))
    saa = np.full_like(S_dn, float(saa))
    
    difvis, difnir, fvis, fnir = py_net_rad.calc_difuse_ratio(S_dn, sza, press=p)

    Skyl = difvis * fvis + difnir * fnir
    S_dn_dir = S_dn * (1.0 - Skyl)
    S_dn_dif = S_dn * Skyl
    rho_leaf_vis = np.full(LAI.shape, rho_vis_C)
    rho_leaf_nir = np.full(LAI.shape, rho_nir_C)
    tau_leaf_vis = np.full(LAI.shape, tau_vis_C)
    tau_leaf_nir = np.full(LAI.shape, tau_nir_C)
    rsoilv = np.full(LAI.shape, rho_vis_S)
    rsoiln = np.full(LAI.shape, rho_nir_S)
    x_lad = np.full(LAI.shape, x_LAD)

    # Calculate the effective LAI (LAI_eff)
    Omega0 = py_clumping.calc_omega0_Kustas(LAI=LAI, f_C=f_c, x_LAD=x_LAD, isLAIeff=True)
    Omega = py_clumping.calc_omega_Kustas(Omega0, sza, w_C=w_C)
    F = LAI / f_c
    LAI_eff = F * Omega

    # Calculate roughness
    z_0M, d_0 = py_net_res.calc_roughness(LAI, h_C, w_C=w_C, landcover=landcover, f_c=f_c)

    # Calculation of canopy and soil shortwave radiation
    Sn_C, Sn_S = py_net_rad.calc_Sn_Campbell(LAI, sza=sza, S_dn_dir=S_dn_dir,
                                            S_dn_dif=S_dn_dif, fvis=fvis, fnir=fnir,
                                            rho_leaf_vis=rho_leaf_vis,
                                            tau_leaf_vis=tau_leaf_vis,
                                            rho_leaf_nir=rho_leaf_nir,
                                            tau_leaf_nir=tau_leaf_nir, rsoilv=rsoilv,
                                            rsoiln=rsoiln, x_LAD=x_LAD, LAI_eff=LAI_eff)
    Sn = Sn_C + Sn_S

    # Run model
    output = run_tseb_model(model_type_to_run,
                            Tr_K=Tr_K if model_type_to_run == 'PT' else None,
                            T_C=T_C, T_S=T_S,
                            vza=vza, T_A_K=T_A_K, u=u, ea=ea, p=p,
                            Sn_C=Sn_C, Sn_S=Sn_S, L_dn=L_dn,
                            LAI=LAI, h_C=h_C,
                            emis_C=emis_C, emis_S=emis_S,
                            z_0M=z_0M, d_0=d_0, z_u=z_u, z_T=z_t,
                            leaf_width=leaf_width, z0_soil=z0_soil, alpha_PT=alpha_PT,
                            x_LAD=x_LAD, f_c=f_c, f_g=f_g, w_C=w_C,
                            resistance_form=[0, {}],
                            calcG_params=[[1], G_ratio],
                            const_L=None, kB=kB,
                            massman_profile=None, verbose=True
                            )
    if model_type_to_run == 'PT':
        [flag, T_S, T_C, T_AC, L_nS, L_nC, LE_C, H_C, LE_S, H_S, G,
         R_S, R_x, R_A, u_friction, L, n_iterations] = output
    elif model_type_to_run == '2T':
        [flag, T_AC, L_nS, L_nC, LE_C, H_C, LE_S, H_S, G,
         R_S, R_x, R_A, u_friction, L, n_iterations] = output
        # Set T_C and T_S manually since they were inputs
        T_C = T_C
        T_S = T_S

    folder_out = df['Output'][loopi]
    os.makedirs(folder_out, exist_ok=True)
    out_path = os.path.join(folder_out, "TSEB.tif")

    # Write to GeoTIFF
    driver = gdal.GetDriverByName('GTiff')
    ds = driver.Create(out_path, dims[1], dims[0], 18, gdal.GDT_Float32)
    ds.SetGeoTransform(gt)
    ds.SetProjection(proj)

    # Output raster bands
    net_longwave = L_nS + L_nC
    total_energy = net_longwave + Sn
    latent_total = LE_C + LE_S
    sensible_total = H_C + H_S
    le_fraction = np.where(latent_total != 0, LE_C / latent_total, -9999)

    band_outputs = [
        clean_band(total_energy), clean_band(sensible_total), clean_band(latent_total),
        clean_band(G), clean_band(le_fraction),
        clean_band(LE_C), clean_band(LE_S),
        clean_band(H_C), clean_band(H_S),
        clean_band(T_C), clean_band(T_S),
        clean_band(net_longwave), clean_band(Sn),
        clean_band(R_S), clean_band(R_x), clean_band(R_A),
        clean_band(u_friction), clean_band(L),
    ]

    for i, band_data in enumerate(band_outputs, 1):
        band = ds.GetRasterBand(i)
        band.WriteArray(band_data)
        band.SetNoDataValue(-9999)
        band.FlushCache()

    ds = None  # Close dataset

print("All TSEB model runs finished.")

c:\Users\A02424951\AppData\Local\anaconda3\envs\pyTSEB\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\clumping_index.py:81: RuntimeWarning: divide by zero encountered in divide
  F = LAI / f_C
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\clumping_index.py:81: RuntimeWarning: overflow encountered in divide
  F = LAI / f_C
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\clumping_index.py:81: RuntimeWarning: invalid value encountered in divide
  F = LAI / f_C
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\clumping_index.py:88: RuntimeWarning: divide by zero encountered in divide
  omega0 = -np.log(trans) / (F * K_be)
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\clumping_index.py:88: RuntimeWarning: invalid value encountered in divide
  omega0 = -np

Iteration: 0, non-converged pixels: 76713, max L diff: inf, total time: 0.000187, loop time: 0.000187
Iteration: 1, non-converged pixels: 74283, max L diff: inf, total time: 0.149463, loop time: 0.149276


c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\TSEB.py:3134: RuntimeWarning: overflow encountered in power
  delta_T_C = (((T_R_K_4 - f_theta * T_C_lin**4 - (1.0 - f_theta) * T_D**4)


Iteration: 2, non-converged pixels: 74222, max L diff: 22.968796, total time: 0.336785, loop time: 0.187322
Iteration: 3, non-converged pixels: 74202, max L diff: 0.693426, total time: 0.520997, loop time: 0.184211
Iteration: 4, non-converged pixels: 74189, max L diff: 7.326916, total time: 0.701905, loop time: 0.180909
Iteration: 5, non-converged pixels: 74114, max L diff: 0.207843, total time: 0.887073, loop time: 0.185168
Iteration: 6, non-converged pixels: 61194, max L diff: 0.120262, total time: 1.024281, loop time: 0.137208
Iteration: 7, non-converged pixels: 10906, max L diff: 4.168915, total time: 1.164273, loop time: 0.139992
Iteration: 8, non-converged pixels: 20, max L diff: 0.011716, total time: 1.211581, loop time: 0.047307
Iteration: 9, non-converged pixels: 6, max L diff: 0.012163, total time: 1.229514, loop time: 0.017934
Iteration: 10, non-converged pixels: 3, max L diff: 0.008744, total time: 1.240304, loop time: 0.010789
Iteration: 11, non-converged pixels: 3, max L 

C:\Users\A02424951\AppData\Local\Temp\ipykernel_2684\4011412444.py:179: RuntimeWarning: invalid value encountered in add
  net_longwave = L_nS + L_nC


Iteration: 0, non-converged pixels: 94715, max L diff: inf, total time: 0.000161, loop time: 0.000161
Iteration: 1, non-converged pixels: 92969, max L diff: inf, total time: 0.145257, loop time: 0.145095


c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\TSEB.py:861: RuntimeWarning: invalid value encountered in subtract
  H_S[i] = rho[i] * c_p[i] * (T_S[i] - T_AC[i]) / R_S[i]
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\TSEB.py:809: RuntimeWarning: invalid value encountered in subtract
  "deltaT": T_S[i] - T_AC[i], 'u': u[i], 'rho': rho[i],
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\net_radiation.py:357: RuntimeWarning: invalid value encountered in subtract
  L_nS = emisGrd * taudl * L_dn + emisGrd * (1.0 - taudl) * L_C - L_S
c:\Copy_Paste\ExtrafilesRui\pyTSEB-master-loop\pyTSEB\net_radiation.py:358: RuntimeWarning: invalid value encountered in subtract
  L_nC = (1 - albl) * (1.0 - taudl) * (L_dn + L_S) - 2.0 * (1.0 - taudl) * L_C


Iteration: 2, non-converged pixels: 92891, max L diff: inf, total time: 0.326393, loop time: 0.181136
Iteration: 3, non-converged pixels: 92867, max L diff: 8.911574, total time: 0.484748, loop time: 0.158355
Iteration: 4, non-converged pixels: 92853, max L diff: 4.417319, total time: 0.640812, loop time: 0.156065
Iteration: 5, non-converged pixels: 92774, max L diff: 8.034993, total time: 0.799104, loop time: 0.158292
Iteration: 6, non-converged pixels: 86451, max L diff: 3.347507, total time: 0.957352, loop time: 0.158248
Iteration: 7, non-converged pixels: 37913, max L diff: 7.512737, total time: 1.125932, loop time: 0.168580
Iteration: 8, non-converged pixels: 769, max L diff: 3.295632, total time: 1.208784, loop time: 0.082852
Iteration: 9, non-converged pixels: 41, max L diff: 7.175172, total time: 1.221328, loop time: 0.012544
Iteration: 10, non-converged pixels: 27, max L diff: 3.300939, total time: 1.231945, loop time: 0.010617
Iteration: 11, non-converged pixels: 23, max L di

C:\Users\A02424951\AppData\Local\Temp\ipykernel_2684\4011412444.py:183: RuntimeWarning: invalid value encountered in divide
  le_fraction = np.where(latent_total != 0, LE_C / latent_total, -9999)
